# ==============================================================================
# 🍄 TOXICOLOGY & BOTANICAL SAFETY: MUSHROOM EDIBILITY VIA BERNOULLI NAIVE BAYES
# ==============================================================================
### Core Principles:
Botanical traits (cap color, odor, gill spacing, veil type) are categorical discrete attributes.
By one-hot encoding each categorical trait into binary indicators:
$$x_{j, \text{trait}} = \begin{cases} 1 & \text{if trait is present} \\ 0 & \text{otherwise} \end{cases}$$
Bernoulli Naive Bayes models the joint likelihood of toxic vs edible fungal characteristics.
Absence of foul odor (`odor=foul`) actively reduces the probability of toxicity!


In [1]:
# STEP 1: IMPORTS & ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, f1_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Botanical Toxicology Environment ready.")


✅ STEP 1: Botanical Toxicology Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET EXPLORATION
Loading Kaggle UCI Mushroom Dataset (8,124 fungal records with 22 categorical attributes).


In [2]:
# STEP 2 & 3: LOAD & SANITIZE
data_path = 'data/mushroom/mushrooms.csv'
df = pd.read_csv(data_path)

df.columns = [c.strip().lower().replace('-', '_') for c in df.columns]
print(f"📊 Dataset Shape: {df.shape[0]:,} fungal specimens, {df.shape[1]} attributes")
print(f"Class Distribution: {df['class'].value_counts().to_dict()}")
df.head(3)


📊 Dataset Shape: 8,124 fungal specimens, 23 attributes
Class Distribution: {'e': 4208, 'p': 3916}


,class,cap_shape,cap_surface,cap_color,bruises,odor,gill_attachment,gill_spacing,gill_size,gill_color,...,stalk_surface_below_ring,stalk_color_above_ring,stalk_color_below_ring,veil_type,veil_color,ring_number,ring_type,spore_print_color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features $X$ and target $y$ (`p` = poisonous -> 1, `e` = edible -> 0).


In [3]:
# STEP 4 & 5: SEGREGATION & TARGET MAPPING
X = df.drop(columns=['class'])
y = df['class'].map({'p': 1, 'e': 0})  # 1 = Poisonous, 0 = Edible

print(f"Missing attributes: {X.isnull().sum().sum()}")
print(f"Target distribution (1=Poisonous):\n{y.value_counts(normalize=True).round(4) * 100}")


Missing attributes: 0
Target distribution (1=Poisonous):
class
0    51.8
1    48.2
Name: proportion, dtype: float64


## 🔒 STEP 6: STRATIFIED SPLIT
80/20 train/test partition.


In [4]:
# STEP 6: SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train Specimens: {X_train.shape[0]:,}")
print(f"Test Specimens : {X_test.shape[0]:,}")


Train Specimens: 6,499
Test Specimens : 1,625


## ⚙️ STEP 7: ONE-HOT ENCODING FOR BERNOULLI NAIVE BAYES
Transforming discrete categories into binary presence indicator columns ($x \in \{0, 1\}$).


In [5]:
# STEP 7: ONE-HOT ENCODER
encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
print("✅ STEP 7: Binary Indicator Encoder configured.")


✅ STEP 7: Binary Indicator Encoder configured.


## ⚠️ STEP 8: BASELINE PIPELINE
Training BernoulliNB on one-hot botanical indicators.


In [6]:
# STEP 8: BASELINE FIT
pipe_base = Pipeline([
    ('ohe', encoder),
    ('bnb', BernoulliNB(binarize=None, alpha=1.0))
])
pipe_base.fit(X_train, y_train)

y_pred_base = pipe_base.predict(X_test)
print(f"Baseline BernoulliNB Accuracy: {accuracy_score(y_test, y_pred_base)*100:.2f}%")
print(f"Baseline BernoulliNB ROC-AUC : {roc_auc_score(y_test, pipe_base.predict_proba(X_test)[:, 1])*100:.2f}%")


Baseline BernoulliNB Accuracy: 93.42%
Baseline BernoulliNB ROC-AUC : 99.57%


## 🎯 STEP 9: TUNING SMOOTHING PARAMETER $\alpha$
Optimizing Laplace smoothing with 5-Fold Stratified Cross-Validation.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'bnb__alpha': np.logspace(-3, 1, 15)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    pipe_base,
    param_grid=param_grid,
    cv=cv,
    scoring='recall',  # Prioritize zero false negatives for poisonous mushrooms!
    n_jobs=-1
)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Alpha Parameter: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV Recall: {grid.best_score_*100:.2f}%")


🏆 Best Alpha Parameter: {'bnb__alpha': np.float64(0.001)}
🎯 Best 5-Fold CV Recall: 98.21%


## ⚔️ STEP 10: ALGORITHM DUEL (BernoulliNB vs Logistic Regression)
Evaluating BernoulliNB against Logistic Regression.


In [8]:
# STEP 10: MODEL BENCHMARK
pipe_lr = Pipeline([
    ('ohe', encoder),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])
pipe_lr.fit(X_train, y_train)

y_pred_bnb = best_model.predict(X_test)
y_pred_lr = pipe_lr.predict(X_test)

print("="*65)
print(f"BernoulliNB (Tuned) -> Test Accuracy: {accuracy_score(y_test, y_pred_bnb)*100:.2f}%, ROC-AUC: {roc_auc_score(y_test, best_model.predict_proba(X_test)[:, 1])*100:.2f}%")
print(f"Logistic Regression -> Test Accuracy: {accuracy_score(y_test, y_pred_lr)*100:.2f}%, ROC-AUC: {roc_auc_score(y_test, pipe_lr.predict_proba(X_test)[:, 1])*100:.2f}%")
print("="*65)


BernoulliNB (Tuned) -> Test Accuracy: 98.83%, ROC-AUC: 99.93%
Logistic Regression -> Test Accuracy: 99.88%, ROC-AUC: 100.00%


## 📊 STEP 11: BOTANICAL SAFETY & ZERO FALSE NEGATIVE AUDIT
Evaluating safety critical toxicity detection.


In [9]:
# STEP 11: EVALUATION REPORT
print("📋 MUSHROOM TOXICITY CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred_bnb, target_names=['Edible', 'Poisonous']))

cm = confusion_matrix(y_test, y_pred_bnb)
print(f"Confusion Matrix:\n{cm}")


📋 MUSHROOM TOXICITY CLASSIFICATION REPORT:
              precision    recall  f1-score   support

      Edible       0.99      0.99      0.99       842
   Poisonous       0.99      0.98      0.99       783

    accuracy                           0.99      1625
   macro avg       0.99      0.99      0.99      1625
weighted avg       0.99      0.99      0.99      1625

Confusion Matrix:
[[835   7]
 [ 12 771]]


## 💾 STEP 12 & 13: SERIALIZATION & FIELD SMOKE TEST
Serializing production model and testing mobile field inference latency.


In [10]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/mushroom_edibility_bernoulli_nb_pipeline.joblib'
joblib.dump(best_model, model_path)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_specimen = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_toxic = loaded_pipe.predict(sample_specimen)[0]
toxic_prob = loaded_pipe.predict_proba(sample_specimen)[0][1]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🍄 LIVE BOTANICAL FORAGING AUDIT:")
print(f"   Specimen Assessment : {'🚨 POISONOUS / TOXIC' if pred_toxic == 1 else '✅ EDIBLE / SAFE'}")
print(f"   Toxicity Probability: {toxic_prob*100:.2f}%")
print(f"   Mobile Latency      : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/mushroom_edibility_bernoulli_nb_pipeline.joblib (10,145 bytes)

🍄 LIVE BOTANICAL FORAGING AUDIT:
   Specimen Assessment : 🚨 POISONOUS / TOXIC
   Toxicity Probability: 100.00%
   Mobile Latency      : 12.036 ms (SLA < 2.0ms: CHECK)
